In [1]:
import pandas as pd
import numpy as np
import gzip
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pickle
import math
from scipy.stats import linregress
import os
import sys
from datetime import date
from sklearn.metrics import auc, f1_score
import random
from sklearn.decomposition import NMF, non_negative_factorization
import seaborn as sns
import time
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1 import axes_grid
import mpl_scatter_density
from matplotlib.colors import LinearSegmentedColormap
import pyBigWig

In [2]:
cycle0_dhs_arr = np.load('/home/ctrader/Hold/DHS/dat_bin_FDR01_hg38_bool_array.npy')

filename = '/home/ctrader/Hold/NMF/DHS/fill_in_multiple_at_a_time_saves/delta_fill_in_multiple_at_a_time_booldhsarr_basis_mixture_fillsize100unique_cycle1_CT20220301.npz'
np_file = np.load(filename)
cycle1_dhs_arr = np_file['new_dhs_array']

In [3]:
#loading biosample metadata
tsv_file = open("/home/ctrader/Hold/DHS/DHS_Index_and_Vocabulary_metadata.tsv")
biosample_df = pd.read_csv(tsv_file, delimiter="\t")

biosample_df['Replicate indicators'] = biosample_df['Replicate indicators'].astype(str)

In [4]:
#find replicates
biosamples_repnames = np.array(biosample_df['Replicate indicators'])
biosamples_sampnames = np.array(biosample_df['Altius Biosample ID'])

In [5]:
cycle0_dhs_int_arr = cycle0_dhs_arr.astype(int)
cycle1_dhs_int_arr = cycle1_dhs_arr.astype(int)

In [6]:
def test_similarity(dhs_arr=cycle0_dhs_arr.astype(int), repnames_arr=biosamples_repnames, sampnames_arr=biosamples_sampnames):
    total_changes = []
    samp_rep_ind_list = []
    for rep_name, rep_ind in zip(repnames_arr, range(len(repnames_arr))):
        if rep_name!='nan':
            if len(np.where(sampnames_arr==rep_name)[0]) == 1:
                samp_ind = np.where(sampnames_arr==rep_name)[0][0]
                if tuple(np.sort(np.array([rep_ind, samp_ind]))) not in samp_rep_ind_list:
                    samp_rep_ind_list.append(tuple(np.sort(np.array([rep_ind, samp_ind]))))
                    changes = np.sum(np.abs(dhs_arr[rep_ind, :] - dhs_arr[samp_ind, :]))
                    total_changes.append(changes)
    total_changes = np.array(total_changes)
    
    average_changes = np.mean(total_changes)
    return average_changes



print(test_similarity(cycle0_dhs_int_arr))
print(test_similarity(cycle1_dhs_int_arr))

99262.1616766467
99216.7365269461


In [7]:
del cycle0_dhs_int_arr
del cycle1_dhs_int_arr
del cycle0_dhs_arr
del cycle1_dhs_arr

In [8]:
for i in range(1, 11):
    filename = '/home/ctrader/Hold/NMF/DHS/fill_in_multiple_at_a_time_saves/delta_fill_in_multiple_at_a_time_booldhsarr_basis_mixture_fillsize100unique_cycle'+str(i)+'_CT20220301.npz'
    np_file = np.load(filename)
    cycle_dhs_arr = np_file['new_dhs_array'].astype(int)
    print('Cycle: ' + str(i) + ' average differences - ' + str(test_similarity(cycle_dhs_arr)))

Cycle: 1 average differences - 99216.7365269461
Cycle: 2 average differences - 99216.64670658682
Cycle: 3 average differences - 99216.50898203593
Cycle: 4 average differences - 99216.4371257485
Cycle: 5 average differences - 99216.19760479042
Cycle: 6 average differences - 99216.03592814371
Cycle: 7 average differences - 99215.92814371258
Cycle: 8 average differences - 99215.8383233533
Cycle: 9 average differences - 99215.73053892216
Cycle: 10 average differences - 99215.64071856288


**Replicate Sanity Check**

In [67]:
total_changes = []
samp_rep_ind_list = []
count=0
tup_count=0
tup_list = []
for rep_name, rep_ind in zip(biosamples_repnames, range(len(biosamples_repnames))):
    if rep_name!='nan':
        if len(np.where(biosamples_sampnames==rep_name)[0]) == 1:
            samp_ind = np.where(biosamples_sampnames==rep_name)[0][0]
            if tuple(np.sort(np.array([rep_ind, samp_ind]))) not in samp_rep_ind_list:
                samp_rep_ind_list.append(tuple(np.sort(np.array([rep_ind, samp_ind]))))
                changes = np.sum(np.abs(cycle0_dhs_int_arr[rep_ind, :] - cycle0_dhs_int_arr[samp_ind, :]))
                total_changes.append(changes)
            else:
                tup_count+=1
                tup_list.append(tuple(np.sort(np.array([rep_ind, samp_ind]))))
        else:
            count+=1
total_changes = np.array(total_changes)
print(total_changes.shape)
average_changes = np.mean(total_changes)

(167,)


In [70]:
for i in samp_rep_ind_list:
    if i not in tup_list:
        print(i)

(56, 158)


In [66]:
tup_count

166

In [76]:
biosample_df.loc[[36, 37]]

,library order,Biosample name,Vocabulary representative,DCC Experiment ID,DCC Library ID,DCC Biosample ID,Altius Aggregation ID,Altius Library ID,Altius Biosample ID,Replicate indicators,...,Library cleanup,DNaseI units/mL,Amount Nucleic Acid (ng),Nuclei count,Protease inhibitor,Library sequencing date,Reads used,DCC SPOT score,Per-biosample peaks,DHSs in Index
36,37,HGF,NaN,ENCSR000ENS,ENCLB486ZZZ,ENCBS786ANZ,AG4192,LN2304,DS11738,DS11752,...,minElute,80.0,5.6,NaN,NaN,2009-08-05,22293070,0.4847,63915,63851
37,38,HGF,NaN,ENCSR000ENS,ENCLB487ZZZ,ENCBS310AAA,AG4191,LN2307,DS11752,DS11738,...,minElute,60.0,9.8,NaN,NaN,2009-08-05,351870884,0.4918,200730,200524


In [71]:
biosample_df.loc[[56, 158]]

,library order,Biosample name,Vocabulary representative,DCC Experiment ID,DCC Library ID,DCC Biosample ID,Altius Aggregation ID,Altius Library ID,Altius Biosample ID,Replicate indicators,...,Library cleanup,DNaseI units/mL,Amount Nucleic Acid (ng),Nuclei count,Protease inhibitor,Library sequencing date,Reads used,DCC SPOT score,Per-biosample peaks,DHSs in Index
56,57,CD34,Myeloid / erythroid,ENCSR191EII,ENCLB262HGC,ENCBS912YBQ,AG9156,LN2482,DS12274,DS14197,...,minElute,80.0,18.5,NaN,NaN,NaN,211609789,0.6518,196198,195691
158,159,CD34,NaN,ENCSR191EII,ENCLB323HYV,ENCBS912YBQ,AG9136,LN2999,DS14197,nan,...,minElute,160.0,8.1,6300000.0,Buffer A,NaN,24389674,0.6518,52106,52010


In [77]:
count + 167*2 - 1

349

In [31]:
biosamples_repnames[biosamples_repnames!='nan'].shape

(349,)